# Multi-Head Attention (MHA)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        B, T, _ = Q.shape
        H, d_k = self.num_heads, self.d_k

        # project + split heads: (B, T, d_model) -> (B, H, T, d_k)
        Q = self.W_q(Q).view(B, T, H, d_k).transpose(1, 2)
        K = self.W_k(K).view(B, -1, H, d_k).transpose(1, 2)
        V = self.W_v(V).view(B, -1, H, d_k).transpose(1, 2)

        # scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (B, H, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ V  # (B, H, T, d_k)

        # merge heads + project
        out = out.transpose(1, 2).contiguous().view(B, T, H * d_k)
        return self.W_o(out)

## 验证

In [ ]:
B, T, d_model, num_heads = 2, 10, 64, 8
x = torch.randn(B, T, d_model)

mha = MultiHeadAttention(d_model, num_heads)
out = mha(x, x, x)

print(f"input:  {x.shape}")
print(f"output: {out.shape}")  # 应为 (2, 10, 64)

## 与 nn.MultiheadAttention 对比 shape

In [ ]:
ref = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
ref_out, _ = ref(x, x, x)

print(f"my MHA output shape:  {out.shape}")
print(f"torch MHA output shape: {ref_out.shape}")
print(f"shape match: {out.shape == ref_out.shape}")